In [ ]:
from rich.console import Console
from dotenv import load_dotenv
load_dotenv(override=True)
from agents import Agent, Runner, trace, function_tool, SQLiteSession
from openai.types.responses import ResponseTextDeltaEvent

In [ ]:
checklist, completed = [], []

In [ ]:
def show(text: str) -> None:
    try:
        Console().print(text)
    except Exception:
        print(text)    

In [ ]:
def get_checklist_report():
    result = ""
    for index, task in enumerate(checklist):
        if completed[index]:
            result += f"Task {index + 1}:  [green][strike]{task}[/green][/strike]\n"
        else:
            result += f"Task {index + 1}:  {task}\n"

    show(result)        
    return result

In [ ]:
@function_tool
def create_checklist(descriptions: list[str]) -> str:
    "Creates a checklist of tasks to be completed from descriptions."

    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

    



In [ ]:
@function_tool
def mark_task_completed(task_number: int, completion_notes: str) -> str:
    "Marks a task as completed in the checklist with completion notes."

    if 1 <= task_number <= len(checklist):
        completed[task_number - 1] = True
        report = get_checklist_report()
        show(completion_notes)
        return report
    else:
        return "Invalid task number."

In [ ]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.

First, create the complete checklist by calling create_checklist exactly once.

Then complete checklist tasks strictly one at a time and in ascending order.
After completing one task, call mark_task_completed exactly once with:
- the number of that task;
- a concise explanation of how it was completed.

Never call multiple tools in parallel.
Do not start the next task until the previous task has been marked completed.

After all tasks are completed, provide the final answer.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""

In [ ]:
import os

from agents import ModelSettings, OpenAIChatCompletionsModel
from openai import AsyncOpenAI

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)
openrouter_model = OpenAIChatCompletionsModel(model="openai/gpt-4o-mini", openai_client=openrouter_client)


agent = Agent(name="Checklist Assistant", instructions=system_message, model=openrouter_model, tools=[create_checklist, mark_task_completed],model_settings=ModelSettings(
        parallel_tool_calls=False
    ),)

In [ ]:
checklist, completed = [], []

In [ ]:
response =  Runner.run_streamed(agent, user_message)

async for event in response.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)